# 🧪 03 — Data Preparation
## Insurance Retention Analytics

> **Objetivo:** Transformar el dataset crudo en un dataset ML-ready, aplicando feature engineering basado en los insights del EDA, encoding de variables categóricas y un train/test split correctamente estratificado.

**Fase CRISP-DM:** Data Preparation  
**Fecha:** 2026

---

## Tabla de contenidos

1. [Setup y carga](#1-setup)
2. [Identificación de variables a excluir (data leakage)](#2-leakage)
3. [Feature engineering: 5 nuevas variables derivadas](#3-fe)
4. [Definición de X (features) e y (target)](#4-xy)
5. [Encoding de variables categóricas](#5-encoding)
6. [Train/test split estratificado](#6-split)
7. [Validación post-split](#7-validacion)
8. [Exportación a Parquet (ML-ready)](#8-export)
9. [Conclusión](#9-conclusion)

---

<a id="1-setup"></a>
## 1. Setup y carga del dataset

Recargamos el dataset original. **Buena práctica:** nunca modificar el archivo crudo (`data/raw/`); todas las transformaciones se generan a partir de él y se guardan en `data/processed/`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Carga
df = pd.read_csv("../data/raw/insurance_policyholder_churn_synthetic.csv")
print(f"📦 Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head(3)

📦 Dataset cargado: 50,000 filas × 40 columnas


,customer_id,as_of_date,region_name,age,age_band,marital_status,customer_tenure_months,multi_policy_flag,num_policies,policy_type,renewal_month,current_premium,premium_last_year,premium_change_pct,num_price_increases_last_3y,coverage_amount,premium_to_coverage_ratio,payment_frequency,autopay_enabled,late_payment_count_12m,missed_payment_flag,payment_method_change_flag,num_claims_12m,num_approved_claims_12m,num_rejected_claims_12m,num_pending_claims_12m,avg_claim_amount,total_claim_amount_12m,total_payout_amount_12m,payout_ratio_12m,avg_settlement_time_days,days_since_last_claim,num_contacts_12m,complaint_flag,complaint_resolution_days,quote_requested_flag,coverage_downgrade_flag,churn_flag,churn_type,churn_probability_true
0,1,2025-12-31,Manawatu-Whanganui,24,18-24,Married,128,1,4,Auto,8,"1,013.4700","1,060.0400",-0.0365,1,"8,924.0900",0.1136,Monthly,1,0,0,0,0,0,0,0,"5,250.1900",0.0000,0.0000,0.7500,7,1616,0,0,0,0,0,0,No churn,0.0486
1,2,2025-12-31,Auckland,70,65-74,Married,76,1,3,Auto,3,"1,126.9000","1,052.2700",0.0479,2,"43,734.4500",0.0258,Monthly,1,0,0,0,0,0,0,0,"4,141.4100",0.0000,0.0000,0.7500,15,1760,1,0,0,0,0,0,No churn,0.0679
2,3,2025-12-31,Bay of Plenty,62,55-64,Married,129,0,1,Auto,5,984.7000,874.2900,0.1184,1,"23,152.3700",0.0425,Annual,0,0,0,0,0,0,0,0,"1,956.4200",0.0000,0.0000,0.7500,24,1581,1,0,0,0,0,0,No churn,0.2238


<a id="2-leakage"></a>
## 2. Identificación de variables a excluir

### ¿Qué es Data Leakage?

> **Data leakage** ocurre cuando el modelo recibe durante el entrenamiento información que **no estaría disponible en el momento real de la predicción**. Esto infla artificialmente las métricas y produce modelos inútiles en producción.

### Variables a excluir y por qué

| Variable | Motivo de exclusión |
|---|---|
| `customer_id` | Identificador único, no es feature predictiva |
| `as_of_date` | Constante (todos = `2025-12-31`), no aporta varianza |
| `churn_type` | Derivada del propio `churn_flag` → **leakage directo** |
| `churn_probability_true` | Es la probabilidad REAL usada para generar las etiquetas → **leakage masivo** |

> 💡 **Por qué importa esto en una entrevista:** este es uno de los errores más comunes en proyectos junior. Detectarlo y explicarlo demuestra madurez técnica.

In [2]:
COLS_LEAKAGE_O_ID = [
    "customer_id",              # ID, no predictor
    "as_of_date",               # constante
    "churn_type",               # derivada del target
    "churn_probability_true",   # probabilidad real → leakage
]

# Validamos que efectivamente as_of_date es constante
print(f"Valores únicos en as_of_date: {df['as_of_date'].nunique()}")
print(f"   → {df['as_of_date'].unique()}")
print(f"\nValores únicos en churn_type:")
print(df['churn_type'].value_counts())

Valores únicos en as_of_date: 1
   → ['2025-12-31']

Valores únicos en churn_type:
churn_type
No churn    34917
Other        8347
Price        5669
Service       654
Claims        214
Payment       199
Name: count, dtype: int64


<a id="3-fe"></a>
## 3. Feature engineering: 5 nuevas variables derivadas

Basándonos en los insights del EDA, creamos **features que capturan patrones de negocio** que las variables crudas no expresan directamente. Esto es lo que distingue a un buen modelo de uno mediocre.

| # | Nueva variable | Definición | Justificación de negocio |
|---|---|---|---|
| 1 | `premium_shock` | 1 si `premium_change_pct > 0.10` | Captura "shock de precio" — gatillo clásico de churn (validado en H7) |
| 2 | `is_new_customer` | 1 si `customer_tenure_months < 12` | El segmento más volátil (62.5% churn) requiere bandera explícita |
| 3 | `is_loyal_customer` | 1 si `customer_tenure_months >= 60` | Bandera de cliente "estable" (5+ años) |
| 4 | `rejected_claim_ratio` | `num_rejected_claims_12m / (num_claims_12m + 1)` | Captura experiencia negativa con siniestros |
| 5 | `risk_score` | Suma de flags de riesgo | Indicador compuesto interpretable para negocio |

> ⚠️  **El `+1` en el denominador** del `rejected_claim_ratio` evita división por cero cuando un cliente no tiene claims. Es una técnica clásica de smoothing.

In [3]:
# ------------------------------------------------------------
# Copia para no mutar el original
# ------------------------------------------------------------
df_fe = df.copy()

# ------------------------------------------------------------
# Feature 1: premium_shock — aumento >10% en la prima
# ------------------------------------------------------------
df_fe["premium_shock"] = (df_fe["premium_change_pct"] > 0.10).astype(int)

# ------------------------------------------------------------
# Feature 2: is_new_customer — antigüedad menor a 1 año
# ------------------------------------------------------------
df_fe["is_new_customer"] = (df_fe["customer_tenure_months"] < 12).astype(int)

# ------------------------------------------------------------
# Feature 3: is_loyal_customer — antigüedad de 5 años o más
# ------------------------------------------------------------
df_fe["is_loyal_customer"] = (df_fe["customer_tenure_months"] >= 60).astype(int)

# ------------------------------------------------------------
# Feature 4: rejected_claim_ratio — % de claims rechazados (con smoothing)
# ------------------------------------------------------------
df_fe["rejected_claim_ratio"] = (
    df_fe["num_rejected_claims_12m"] / (df_fe["num_claims_12m"] + 1)
).round(4)

# ------------------------------------------------------------
# Feature 5: risk_score — suma de flags de riesgo (interpretable)
# ------------------------------------------------------------
df_fe["risk_score"] = (
    df_fe["missed_payment_flag"]
    + df_fe["complaint_flag"]
    + df_fe["coverage_downgrade_flag"]
    + df_fe["quote_requested_flag"]
    + df_fe["premium_shock"]
    + (df_fe["late_payment_count_12m"] >= 2).astype(int)
)

print("✅ 5 nuevas features creadas")
print(f"   Shape antes : {df.shape}")
print(f"   Shape después: {df_fe.shape}")

✅ 5 nuevas features creadas
   Shape antes : (50000, 40)
   Shape después: (50000, 45)


In [4]:
# Validación rápida: distribución de las nuevas features y su relación con churn
nuevas = ["premium_shock", "is_new_customer", "is_loyal_customer", "rejected_claim_ratio", "risk_score"]

print("📊 Distribución y churn rate de las nuevas features:\n")
for f in nuevas:
    if df_fe[f].dtype in ["int64", "int32"]:
        dist = df_fe[f].value_counts().sort_index()
        churn_by_val = df_fe.groupby(f)["churn_flag"].mean().mul(100).round(2)
        print(f"🔹 {f}:")
        for val in dist.index:
            print(f"   {val} → {dist[val]:>6,} clientes  |  Churn: {churn_by_val[val]:>6.2f}%")
        print()
    else:
        print(f"🔹 {f} (numérica continua): mean = {df_fe[f].mean():.3f}, max = {df_fe[f].max():.3f}\n")

📊 Distribución y churn rate de las nuevas features:

🔹 premium_shock:
   0 → 34,285 clientes  |  Churn:  23.25%
   1 → 15,715 clientes  |  Churn:  45.26%

🔹 is_new_customer:
   0 → 47,060 clientes  |  Churn:  28.02%
   1 →  2,940 clientes  |  Churn:  64.49%

🔹 is_loyal_customer:
   0 → 16,428 clientes  |  Churn:  38.74%
   1 → 33,572 clientes  |  Churn:  25.97%

🔹 rejected_claim_ratio (numérica continua): mean = 0.004, max = 0.667

🔹 risk_score:
   0 → 25,842 clientes  |  Churn:  19.58%
   1 → 16,427 clientes  |  Churn:  34.30%
   2 →  6,236 clientes  |  Churn:  53.09%
   3 →  1,353 clientes  |  Churn:  69.99%
   4 →    128 clientes  |  Churn:  89.84%
   5 →     13 clientes  |  Churn: 100.00%
   6 →      1 clientes  |  Churn: 100.00%



**Observación clave del `risk_score`:**

Esta variable compuesta es **directamente interpretable** para el negocio:

- `risk_score = 0` → cliente sin señales de riesgo
- `risk_score = 1-2` → vigilancia
- `risk_score ≥ 3` → intervención prioritaria

Veamos cómo escala el churn rate con el score:

In [5]:
# El risk_score como predictor interpretable
risk_table = (
    df_fe.groupby("risk_score")["churn_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count":"n_clientes", "mean":"churn_rate"})
)
risk_table["churn_rate"] = risk_table["churn_rate"] * 100
risk_table["pct_portafolio"] = (risk_table["n_clientes"] / len(df_fe) * 100).round(2)
display(risk_table.round(2))

,n_clientes,churn_rate,pct_portafolio
risk_score,,,
0,25842,19.5800,51.6800
1,16427,34.3000,32.8500
2,6236,53.0900,12.4700
3,1353,69.9900,2.7100
4,128,89.8400,0.2600
5,13,100.0000,0.0300
6,1,100.0000,0.0000


<a id="4-xy"></a>
## 4. Definición de X (features) e y (target)

Separamos el dataset en:
- **`y`**: variable objetivo (`churn_flag`)
- **`X`**: todas las demás variables, excluyendo identificadores y variables con leakage

In [6]:
# Variable objetivo
y = df_fe["churn_flag"]

# Variables predictoras (excluimos leakage + target)
X = df_fe.drop(columns=COLS_LEAKAGE_O_ID + ["churn_flag"])

print(f"X shape : {X.shape}  ({X.shape[1]} features)")
print(f"y shape : {y.shape}")
print(f"\nTasa de churn en y: {y.mean()*100:.2f}%")
print(f"\nTipos en X:")
print(X.dtypes.value_counts())

X shape : (50000, 40)  (40 features)
y shape : (50000,)

Tasa de churn en y: 30.17%

Tipos en X:
int64      22
float64    10
object      5
int32       3
Name: count, dtype: int64


<a id="5-encoding"></a>
## 5. Encoding de variables categóricas

Los modelos basados en árboles (Random Forest, XGBoost) no manejan strings directamente. Convertimos las categóricas con **One-Hot Encoding** (`pd.get_dummies`).

### Detalle clave: `drop_first=True`

Si una variable tiene **k categorías**, creamos **k-1 columnas** (no k). La categoría omitida se convierte en la "referencia" (cuando todas las dummies = 0, sabemos que estamos en esa).

> 🎓 **Por qué evitar la "Dummy Variable Trap":**
> Si creáramos k columnas, una sería redundante (combinación lineal exacta de las otras k-1). Esto causa **multicolinealidad perfecta**, que rompe modelos lineales y hace que XGBoost asigne importancia repartida ineficientemente.

> 💡 **Para árboles puros (RF/XGBoost)** el efecto es mínimo, pero es buena práctica universal.

In [7]:
# Identificamos categóricas
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print(f"📑 Columnas categóricas ({len(cat_cols)}):")
for c in cat_cols:
    print(f"   • {c} — {X[c].nunique()} categorías")
print(f"\n📊 Columnas numéricas: {len(num_cols)}")

📑 Columnas categóricas (5):
   • region_name — 7 categorías
   • age_band — 7 categorías
   • marital_status — 2 categorías
   • policy_type — 5 categorías
   • payment_frequency — 2 categorías

📊 Columnas numéricas: 35


In [8]:
# One-Hot Encoding con drop_first=True
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)

print(f"📐 Encoding completado:")
print(f"   Antes  : {X.shape}")
print(f"   Después: {X_encoded.shape}")
print(f"\n📋 Nuevas columnas creadas por encoding:")
nuevas_cols = [c for c in X_encoded.columns if c not in X.columns]
for c in nuevas_cols:
    print(f"   • {c}")

📐 Encoding completado:
   Antes  : (50000, 40)
   Después: (50000, 53)

📋 Nuevas columnas creadas por encoding:
   • region_name_Bay of Plenty
   • region_name_Canterbury
   • region_name_Manawatu-Whanganui
   • region_name_Otago
   • region_name_Waikato
   • region_name_Wellington
   • age_band_25-34
   • age_band_35-44
   • age_band_45-54
   • age_band_55-64
   • age_band_65-74
   • age_band_75+
   • marital_status_Single
   • policy_type_Health
   • policy_type_Home
   • policy_type_Life
   • policy_type_Travel
   • payment_frequency_Monthly


<a id="6-split"></a>
## 6. Train/test split estratificado

### ¿Por qué estratificar?

En clasificación binaria con clases desbalanceadas (70% retenidos, 30% churn), un split aleatorio "puro" puede generar:
- Train con 28% churn
- Test con 32% churn

Esa diferencia parece pequeña, pero **distorsiona la evaluación**. La métrica de test estaría midiendo un mundo distinto al de train.

### Solución: `stratify=y`

Con esta opción, `train_test_split` garantiza que la **proporción de churn sea idéntica** en train y test. Es la práctica estándar profesional.

> 🎓 **`random_state=42`** asegura reproducibilidad: cualquier persona que ejecute el notebook obtiene el mismo split.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.20,        # 80% train, 20% test (estándar)
    stratify=y,            # ⚠️ Estratificar por churn_flag
    random_state=42        # reproducibilidad
)

print(f"📊 Resultado del split estratificado:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test : {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test : {y_test.shape}")

📊 Resultado del split estratificado:
   X_train: (40000, 53)
   X_test : (10000, 53)
   y_train: (40000,)
   y_test : (10000,)


<a id="7-validacion"></a>
## 7. Validación post-split

Validamos que la **proporción de churn se haya preservado** en train y test (debe ser ≈ 30.17% en ambos).

In [10]:
resumen_split = pd.DataFrame({
    "Conjunto": ["Original (full)", "Train (80%)", "Test (20%)"],
    "N clientes": [len(y), len(y_train), len(y_test)],
    "% Churn": [y.mean()*100, y_train.mean()*100, y_test.mean()*100],
    "% Retenido": [(1-y.mean())*100, (1-y_train.mean())*100, (1-y_test.mean())*100],
}).round(2)

display(resumen_split)

# Verificación numérica
diff = abs(y_train.mean() - y_test.mean()) * 100
print(f"\n✅ Diferencia en % churn entre train y test: {diff:.3f} pp")
print("   (Esperado: < 0.05 pp con stratify=y)")

,Conjunto,N clientes,% Churn,% Retenido
0,Original (full),50000,30.1700,69.8300
1,Train (80%),40000,30.1600,69.8400
2,Test (20%),10000,30.1700,69.8300



✅ Diferencia en % churn entre train y test: 0.005 pp
   (Esperado: < 0.05 pp con stratify=y)


<a id="8-export"></a>
## 8. Exportación a Parquet (formato ML-ready)

Guardamos:
- **`churn_ml_ready.parquet`** — dataset completo X+y con features finales
- **`X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`** — splits para reutilizar en el notebook 04 sin tener que rehacer el split

### ¿Por qué Parquet en vez de CSV?

| Aspecto | CSV | Parquet |
|---|---|---|
| Preserva tipos de dato | ❌ todo es texto | ✅ tipos nativos |
| Tamaño en disco | 100% | **~30%** (columnar + compresión) |
| Velocidad de lectura | 1× | **5-10×** |
| Estándar en Big Data | No | Sí (Spark, Athena, BigQuery) |

In [11]:
# Crear carpeta processed
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

# Dataset completo (X + y) con todas las features finales
df_ml_ready = X_encoded.copy()
df_ml_ready["churn_flag"] = y.values
df_ml_ready.to_parquet(out_dir / "churn_ml_ready.parquet", index=False)

# Splits (los reutilizaremos en el notebook 04)
X_train.to_parquet(out_dir / "X_train.parquet", index=False)
X_test.to_parquet(out_dir / "X_test.parquet", index=False)
y_train.to_frame("churn_flag").to_parquet(out_dir / "y_train.parquet", index=False)
y_test.to_frame("churn_flag").to_parquet(out_dir / "y_test.parquet", index=False)

print("✅ Archivos guardados en data/processed/:")
for f in sorted(out_dir.glob("*.parquet")):
    size_kb = f.stat().st_size / 1024
    print(f"   • {f.name:<35} {size_kb:>8,.1f} KB")

✅ Archivos guardados en data/processed/:
   • churn_ml_ready.parquet               2,658.6 KB
   • X_test.parquet                         583.8 KB
   • X_train.parquet                      2,140.6 KB
   • y_test.parquet                           2.6 KB
   • y_train.parquet                          6.6 KB


<a id="9-conclusion"></a>
## 9. Conclusión

### ✅ Lo que dejamos hecho

| Aspecto | Resultado |
|---|---|
| Variables excluidas (ID + leakage) | 4 (`customer_id`, `as_of_date`, `churn_type`, `churn_probability_true`) |
| Features nuevas creadas | 5 (`premium_shock`, `is_new_customer`, `is_loyal_customer`, `rejected_claim_ratio`, `risk_score`) |
| Encoding categóricas | One-Hot Encoding con `drop_first=True` |
| Split | 80% train / 20% test, estratificado por churn |
| Formato de salida | Parquet (5× más rápido que CSV) |
| Reproducibilidad | `random_state=42` fijado |

### 🎓 Conceptos clave aprendidos

1. **Data leakage**: incluir variables derivadas del target rompe el modelo (le da las respuestas).
2. **Feature engineering**: crear variables que capturen patrones de negocio mejora el modelo más que tunear hiperparámetros.
3. **Estratificación**: en clases desbalanceadas, `stratify=y` es obligatorio.
4. **Parquet vs CSV**: estándar profesional para datasets de trabajo.
5. **Smoothing** (`+1` en denominador): truco para evitar divisiones por cero en ratios.

### 🚀 Próximo paso

➡️ **Notebook `04_machine_learning`:** entrenar Random Forest, XGBoost básico, XGBoost balanceado y XGBoost optimizado. Compararemos métricas y seleccionaremos el modelo final.

---

**Fin del notebook 03 — Data Preparation** 🧪